# MedBridge — multilingual medical LoRA on NLLB-200 (Mac)

**Goal:** one adapter for English↔Spanish, English↔Hindi and English↔Mandarin. The sample CSV is **illustrative, unreviewed** and for pipeline smoke testing only. Do not claim clinical accuracy or improvements from it. Never use patient-identifiable data.

**Run this notebook in the MedBridge repository root** (where `server.py` lives), with `illustrative_pairs.csv` beside it. Real training requires a licensed, reviewed parallel corpus with `src_lang,tgt_lang,source,target` columns and a separately held-out evaluation set. Check source licensing and human review for clinical use.

On Apple Silicon, MPS may work; on Intel Macs, CPU training of 600M can be impractically slow. Start with `MAX_STEPS=2` and a tiny batch. This does **not** fine-tune emotion recognition; that is a separate component.

## 0. Install (Terminal, once)
Create a separate environment so the live app remains unaffected. Run the following in Terminal, then choose the new `medbridge-lora` kernel in Jupyter.

In [6]:
# Terminal commands (do not run this cell as Python):
# python3 -m venv .venv-lora
# source .venv-lora/bin/activate
# python -m pip install --upgrade pip
# python -m pip install 'transformers>=4.40,<5' 'peft>=0.10' torch sentencepiece pandas ipykernel notebook
# python -m ipykernel install --user --name medbridge-lora --display-name 'medbridge-lora'
# jupyter notebook

## 1. Load data, create both translation directions, and split by original sentence group

In [1]:
from pathlib import Path
import pandas as pd
import random, torch
random.seed(42); torch.manual_seed(42)
LANG = {'en':'eng_Latn','es':'spa_Latn','hi':'hin_Deva','zh':'zho_Hans'}
DATA_PATH = Path('illustrative_pairs.csv') # replace with your reviewed CSV when ready
SMOKE_TEST = True  # MUST be False for a real training claim
raw = pd.read_csv(DATA_PATH).dropna()
required={'src_lang','tgt_lang','source','target'}
assert required.issubset(raw.columns)
raw = raw.drop_duplicates(subset=['src_lang','tgt_lang','source','target']).reset_index(drop=True)
assert all(x in LANG for x in raw.src_lang) and all(x in LANG for x in raw.tgt_lang)
assert all(raw.src_lang != raw.tgt_lang)
assert all(raw.source.str.strip()) and all(raw.target.str.strip())
print('Original parallel pairs:',len(raw))
print(raw.groupby(['src_lang','tgt_lang']).size())
if not SMOKE_TEST:
    assert len(raw)>=100, 'Need substantially more reviewed pairs for meaningful training'
    assert all(raw.groupby('tgt_lang').size().reindex(['es','hi','zh'],fill_value=0)>=20)

# Split original pairs FIRST, before reversing, to prevent reverse-direction leakage.
indices=list(range(len(raw))); random.shuffle(indices)
cut=max(1,int(len(indices)*0.8)); cut=min(cut,len(indices)-1)
train_original=raw.iloc[indices[:cut]]
val_original=raw.iloc[indices[cut:]]
def bidirectional(frame):
    forward=frame.copy()
    reverse=frame.rename(columns={'src_lang':'tgt_lang','tgt_lang':'src_lang','source':'target','target':'source'})
    return pd.concat([forward,reverse],ignore_index=True)
train=bidirectional(train_original).to_dict('records')
val=bidirectional(val_original).to_dict('records')
print('Train directional examples:',len(train),'Validation:',len(val))
print('Example:',train[0])

Original parallel pairs: 12
src_lang  tgt_lang
en        es          4
          hi          4
          zh          4
dtype: int64
Train directional examples: 18 Validation: 6
Example: {'src_lang': 'en', 'tgt_lang': 'hi', 'source': 'Do you have chest pain?', 'target': 'क्या आपके सीने में दर्द है?'}


## 2. Load NLLB and attach LoRA

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import LoraConfig, TaskType, get_peft_model
MODEL_NAME='facebook/nllb-200-distilled-600M'
DEVICE='mps' if torch.backends.mps.is_available() else 'cpu'
print('Device:',DEVICE)
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
base=AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
base.config.use_cache=False
config=LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM,r=4,lora_alpha=8,lora_dropout=0.05,bias='none',target_modules=['q_proj','v_proj'])
model=get_peft_model(base,config).to(DEVICE)
model.print_trainable_parameters()

/Users/siapatelpuri/Desktop/Sachacks/MedBridge/.venv-lora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
trainable params: 589,824 || all params: 615,663,616 || trainable%: 0.0958


## 3. Encode source and target using the correct NLLB language tags

In [3]:
MAX_LENGTH=80
def encode_example(row):
    tokenizer.src_lang=LANG[row['src_lang']]
    tokenizer.tgt_lang=LANG[row['tgt_lang']]
    item=tokenizer(row['source'],max_length=MAX_LENGTH,truncation=True,return_tensors='pt')
    target=tokenizer(text_target=row['target'],max_length=MAX_LENGTH,truncation=True,return_tensors='pt')
    return {'input_ids':item['input_ids'].to(DEVICE),
            'attention_mask':item['attention_mask'].to(DEVICE),
            'labels':target['input_ids'].to(DEVICE)}
encoded_train=[encode_example(r) for r in train]
encoded_val=[encode_example(r) for r in val]
print('Encoded:',len(encoded_train),'train examples')

Encoded: 18 train examples


## 4. Run a small LoRA training loop (batch size 1)

In [4]:
import time
from torch.optim import AdamW
MAX_STEPS=2 if SMOKE_TEST else 100  # tune after confirming runtime and data quality
LR=2e-4
optimizer=AdamW((p for p in model.parameters() if p.requires_grad),lr=LR)
model.train(); start=time.time()
for step in range(MAX_STEPS):
    sample=encoded_train[step%len(encoded_train)]
    optimizer.zero_grad(set_to_none=True)
    loss=model(**sample).loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad),1.0)
    optimizer.step()
    print(f'Step {step+1}/{MAX_STEPS}: loss={loss.item():.4f} elapsed={time.time()-start:.1f}s')
print('Training finished. This is a smoke test, not a medical-quality model.' if SMOKE_TEST else 'Training finished; evaluate against baseline on held-out data.')

Step 1/2: loss=1.2570 elapsed=6.6s
Step 2/2: loss=1.0076 elapsed=7.3s
Training finished. This is a smoke test, not a medical-quality model.


## 5. Save adapter, test translations, and check validation loss

In [12]:
ADAPTER_DIR=Path('adapters/medical_multilingual_smoke' if SMOKE_TEST else 'adapters/medical_multilingual')
ADAPTER_DIR.mkdir(parents=True,exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved adapter:',ADAPTER_DIR.resolve())
model.eval()
@torch.inference_mode()
def translate(text,src,tgt):
    tokenizer.src_lang=LANG[src]
    x=tokenizer(text,return_tensors='pt',truncation=True,max_length=MAX_LENGTH).to(DEVICE)
    ids=model.generate(**x,forced_bos_token_id=tokenizer.convert_tokens_to_ids(LANG[tgt]),max_new_tokens=80,num_beams=2)
    return tokenizer.batch_decode(ids,skip_special_tokens=True)[0]
for lang in ['es','hi','zh']:
    print(lang,translate('Do not take two tablets at once.','en',lang))
losses=[]
with torch.inference_mode():
    for item in encoded_val:
        losses.append(model(**item).loss.item())
print('Held-out loss (small set; not a clinical metric):',sum(losses)/len(losses))

Saved adapter: /Users/siapatelpuri/Desktop/Sachacks/MedBridge/adapters/medical_multilingual_smoke


/Users/siapatelpuri/Desktop/Sachacks/MedBridge/.venv-lora/lib/python3.12/site-packages/transformers/generation/utils.py:1733: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed in v5. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


es No tome dos comprimidos a la vez.
hi एक ही समय में दो गोलियां न लें।
zh 不要同时服用两种药丸.
Held-out loss (small set; not a clinical metric): 1.1307268192370732


## 6. Verify saved adapter and integrate later

This cell checks that the adapter files exist. It does **not** reload another 600M model into memory. The live `server.py` should use the smoke-test adapter only for development, not clinical demonstrations or medical-accuracy claims.


In [5]:
from pathlib import Path
import json

# ADAPTER_DIR is set in Section 5. This also works after a kernel restart.
adapter_dir = Path(globals().get("ADAPTER_DIR", "adapters/medical_multilingual_smoke"))
config_file = adapter_dir / "adapter_config.json"
weights_file = next((p for p in [adapter_dir / "adapter_model.safetensors", adapter_dir / "adapter_model.bin"] if p.exists()), None)

assert config_file.is_file(), f"Missing adapter config: {config_file}"
assert weights_file is not None, f"Missing adapter weights in {adapter_dir}"

config = json.loads(config_file.read_text())
print("Adapter verified:", adapter_dir.resolve())
print("Base model:", config.get("base_model_name_or_path"))
print("Weights:", weights_file.name)
print("To test in server.py, load the adapter with PeftModel.from_pretrained(base, str(adapter_dir)).")
print("This smoke-test adapter has NOT been clinically validated.")


Adapter verified: /Users/siapatelpuri/Desktop/Sachacks/MedBridge/adapters/medical_multilingual_smoke
Base model: facebook/nllb-200-distilled-600M
Weights: adapter_model.safetensors
To test in server.py, load the adapter with PeftModel.from_pretrained(base, str(adapter_dir)).
This smoke-test adapter has NOT been clinically validated.
